# Cinema 21 AI Agent - Google Colab
**Agentic AI Customer Service** — Gemini Function Calling + RAG + SQLite

Jalankan setiap cell secara berurutan.

## Cell 1 - Install Dependencies (~2-3 menit)

In [ ]:
!pip install -q fastapi uvicorn google-genai faiss-cpu sentence-transformers reportlab pdfminer.six streamlit pyngrok httpx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 78.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 91.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 113.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 85.0 MB/s eta 0:00:00


## Cell 2 - Load API Keys
Tambahkan di Colab Secrets (ikon kunci di sidebar kiri):
- `NGROK_TOKEN` dari ngrok.com (gratis)
- `GEMINI_KEY` dari aistudio.google.com (gratis)

In [ ]:
from pyngrok import ngrok
from google.colab import userdata

ngrok.set_auth_token(userdata.get('NGROK_TOKEN'))
print('Tokens loaded OK')

Tokens loaded OK


## Cell 3 - Write Project Files (database, rag, tools, agent, backend)

In [ ]:
import os
os.makedirs('cinema_agent', exist_ok=True)

DATABASE_PY = '''import sqlite3, os
DB_PATH = "cinema_agent/cinema.db"

def get_conn():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn

def init_db():
    conn = get_conn(); cur = conn.cursor()
    cur.executescript("""
    CREATE TABLE IF NOT EXISTS movies (id INTEGER PRIMARY KEY AUTOINCREMENT,
        title TEXT NOT NULL, genre TEXT, duration INTEGER, age_rating TEXT, description TEXT);
    CREATE TABLE IF NOT EXISTS cinemas (id INTEGER PRIMARY KEY AUTOINCREMENT, name TEXT NOT NULL, location TEXT);
    CREATE TABLE IF NOT EXISTS studios (id INTEGER PRIMARY KEY AUTOINCREMENT,
        cinema_id INTEGER, studio_name TEXT, type TEXT, FOREIGN KEY (cinema_id) REFERENCES cinemas(id));
    CREATE TABLE IF NOT EXISTS showtimes (id INTEGER PRIMARY KEY AUTOINCREMENT,
        movie_id INTEGER, studio_id INTEGER, show_date TEXT, show_time TEXT, price INTEGER,
        FOREIGN KEY (movie_id) REFERENCES movies(id), FOREIGN KEY (studio_id) REFERENCES studios(id));
    CREATE TABLE IF NOT EXISTS seats (id INTEGER PRIMARY KEY AUTOINCREMENT,
        studio_id INTEGER, seat_number TEXT, FOREIGN KEY (studio_id) REFERENCES studios(id));
    CREATE TABLE IF NOT EXISTS bookings (id INTEGER PRIMARY KEY AUTOINCREMENT,
        customer_name TEXT NOT NULL, showtime_id INTEGER, seat_number TEXT,
        status TEXT DEFAULT \'confirmed\',
        created_at TEXT DEFAULT (datetime(\'now\',\'localtime\')),
        FOREIGN KEY (showtime_id) REFERENCES showtimes(id));
    """)
    if cur.execute("SELECT COUNT(*) FROM movies").fetchone()[0] == 0: _seed(cur)
    conn.commit(); conn.close(); print("DB ready.")

def _seed(cur):
    from datetime import date, timedelta
    today = date.today()
    dates = [(today + timedelta(days=i)).isoformat() for i in range(5)]
    movies = [
        ("Avatar: Fire and Ash","Sci-Fi/Action",162,"13+","Jake Sully dan Neytiri kembali."),
        ("Inception 2: Dreamfall","Thriller/Sci-Fi",148,"17+","Dom Cobb kembali ke dunia mimpi."),
        ("Moana 2","Animation/Adventure",100,"SU","Moana memulai perjalanan baru."),
        ("Venom: The Last Dance","Action/Superhero",139,"13+","Eddie Brock pertempuran terakhir."),
        ("Dune: Messiah","Sci-Fi/Epic",155,"13+","Paul Atreides menjadi Kaisar."),
    ]
    cur.executemany("INSERT INTO movies (title,genre,duration,age_rating,description) VALUES (?,?,?,?,?)", movies)
    cur.execute("INSERT INTO cinemas (name,location) VALUES (?,?)",
                ("Cinema 21 Grand Indonesia","Grand Indonesia Mall Lt.3A"))
    cinema_id = cur.lastrowid
    studios = [(cinema_id,"Studio 1","Regular"),(cinema_id,"Studio 2","Regular"),
               (cinema_id,"Studio 3 - IMAX","IMAX"),(cinema_id,"Studio 4 - 4DX","4DX"),
               (cinema_id,"Studio 5 - Premiere","Premiere")]
    cur.executemany("INSERT INTO studios (cinema_id,studio_name,type) VALUES (?,?,?)", studios)

    # FIX: ambil dari DB bukan dari tuple lokal
    studio_rows = cur.execute("SELECT id, type FROM studios").fetchall()

    for sid, stype in studio_rows:
        cur.executemany("INSERT INTO seats (studio_id,seat_number) VALUES (?,?)",
                        [(sid,f"{r}{n}") for r in "ABCDEFGH" for n in range(1,9)])

    prices = {"Regular":50000,"IMAX":100000,"4DX":120000,"Premiere":150000}
    movie_ids = [r[0] for r in cur.execute("SELECT id FROM movies").fetchall()]
    times = ["11:00","14:00","17:00","20:00"]
    data = []
    for d in dates:
        for mid in movie_ids:
            for i, (sid, stype) in enumerate(studio_rows[:3]):  # FIX: dari DB
                data.append((mid, sid, d, times[i % len(times)], prices[stype]))
    cur.executemany("INSERT INTO showtimes (movie_id,studio_id,show_date,show_time,price) VALUES (?,?,?,?,?)", data)
    print("Seed done.")
'''
GENERATE_PDF_PY = '''from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, HRFlowable
from reportlab.lib.enums import TA_JUSTIFY

KNOWLEDGE = """
=== INFORMASI UMUM BIOSKOP ===
Cinema 21 Grand Indonesia berlokasi di Grand Indonesia Mall Lantai 3A, Jakarta Pusat.
Bioskop beroperasi setiap hari mulai pukul 10.00 WIB hingga 23.59 WIB.
Layanan pelanggan: WhatsApp 0800-2100-21.

=== JENIS STUDIO & FASILITAS ===
Studio Regular: Layar standar Dolby Digital, kapasitas 64 kursi, harga Rp 50.000.
Studio IMAX: Teknologi 4K laser projection, suara 12 saluran, harga Rp 100.000.
Studio 4DX: Kursi bergerak, efek angin/air/kilat/kabut, harga Rp 120.000. Tidak untuk ibu hamil.
Studio Premiere: Sofa recliner, welcome drink sudah termasuk, harga Rp 150.000. Min usia 18 tahun.

=== ATURAN TIKET ===
Tiket dapat dibeli di loket atau M-Tix. Pembelian online maksimal 7 hari sebelumnya.
Film rating SU: semua usia. Rating 13+: di bawah 13 tahun harus didampingi. Rating 17+: tidak boleh masuk.
Dilarang: handphone saat film, merekam layar, membawa makanan dari luar, merokok.

=== KEBIJAKAN REFUND DAN PEMBATALAN ===
Pembatalan tiket maksimal 2 jam sebelum jadwal. Biaya admin Rp 5.000 per tiket.
Refund diproses 3-7 hari kerja. Tiket flash sale tidak bisa direfund.
Reschedule maksimal 1 jam sebelum tayang, hanya 1 kali per transaksi.
Jika bioskop membatalkan: refund penuh plus voucher Rp 25.000.

=== MENU MAKANAN DAN MINUMAN ===
Popcorn Caramel M Rp 35.000 / L Rp 45.000. Popcorn Asin M Rp 30.000 / L Rp 40.000.
Soft Drink M Rp 30.000 / L Rp 38.000. Air Mineral Rp 15.000.
Nachos Rp 40.000. Hot Dog Rp 45.000. French Fries Rp 35.000.
Combo 1 (Popcorn M + Soft Drink M) Rp 55.000. Combo 2 (L+L) Rp 70.000.
Makanan dari luar TIDAK diperbolehkan masuk studio.

=== PROGRAM KEANGGOTAAN ===
Membership Rp 50.000/tahun: diskon 20% hari Selasa, poin reward, birthday voucher 1 tiket gratis.
M-Tix Premium Rp 30.000/bulan: 1 tiket gratis per bulan, prioritas pemilihan kursi.

=== PROMO DAN DISKON ===
Selasa Member: diskon 20%. Senin Gembira: Rp 35.000 studio Regular.
Student Price: diskon 15% dengan kartu pelajar.
Buy 2 Get 1: hari kerja jam 10-13. Promo Bank Mandiri/BCA: diskon 25% weekend.
Gopay/OVO: cashback 10%.

=== FAQ ===
Q: Bisakah memilih kursi sendiri? A: Ya, saat pembelian tiket online maupun di loket.
Q: Bagaimana jika terlambat? A: Boleh masuk, tempat duduk tidak dijamin jika terlambat 15+ menit.
Q: Anak kecil perlu tiket? A: Di bawah 3 tahun tidak perlu jika dipangku. Usia 3+ wajib tiket.
Q: Ada kursi disabilitas? A: Ya, hubungi petugas loket.
Q: Bisakah tiket dipindahtangankan? A: Bisa dengan menunjukkan bukti pembelian.

=== INFORMASI DAN SINOPSIS FILM ===

Avatar: Fire and Ash (2025)
Genre: Sci-Fi/Action | Durasi: 162 menit | Rating: 13+
Sinopsis: Jake Sully dan Neytiri kembali menghadapi ancaman baru dari penjuru galaksi yang mengincar Pandora. Pertempuran epik antara manusia dan Na\'vi memasuki babak baru yang lebih intens dan emosional.
Cocok untuk: Pecinta aksi, visual spektakuler, science fiction. Sangat direkomendasikan di studio IMAX atau 4DX.
Tidak cocok untuk: Anak di bawah 13 tahun tanpa pendampingan.

Inception 2: Dreamfall (2025)
Genre: Thriller/Sci-Fi | Durasi: 148 menit | Rating: 17+
Sinopsis: Dom Cobb terpaksa kembali ke dunia mimpi untuk menyelamatkan putranya yang terjebak di lapisan terdalam kesadaran. Lebih kompleks dan gelap dari film pertama.
Cocok untuk: Penonton yang suka film mind-bending, psychological thriller, cerita berlapis.
Tidak cocok untuk: Anak di bawah 17 tahun, penonton yang ingin film ringan.

Moana 2 (2025)
Genre: Animation/Adventure | Durasi: 100 menit | Rating: SU
Sinopsis: Moana memulai perjalanan baru melewati samudra tak berujung untuk menemukan leluhur yang hilang. Penuh lagu, warna, dan semangat petualangan keluarga.
Cocok untuk: Semua usia, keluarga dengan anak-anak, film ringan dan menyenangkan.

Venom: The Last Dance (2025)
Genre: Action/Superhero | Durasi: 139 menit | Rating: 13+
Sinopsis: Eddie Brock dan Venom menghadapi pertempuran terakhir yang menentukan nasib simbiote di seluruh dunia. Penutup trilogi yang penuh aksi dan humor gelap.
Cocok untuk: Penggemar Marvel, film superhero, aksi intens.
Catatan: Disarankan nonton Venom 1 dan 2 terlebih dahulu.

Dune: Messiah (2025)
Genre: Sci-Fi/Epic | Durasi: 155 menit | Rating: 13+
Sinopsis: Paul Atreides kini menjadi Kaisar Arrakis namun kekuasaan membawa harga yang tidak terduga. Film epik dengan visual dan cerita yang sangat mendalam.
Cocok untuk: Penggemar Dune Part 1 dan 2, sci-fi serius, film epik panjang.
Tidak cocok untuk: Penonton yang belum nonton Dune sebelumnya.
"""

def generate_pdf(output_path="cinema_agent/cinema_knowledge.pdf"):
    doc = SimpleDocTemplate(output_path, pagesize=A4,
                            rightMargin=2*cm, leftMargin=2*cm, topMargin=2*cm, bottomMargin=2*cm)
    styles = getSampleStyleSheet()
    h = ParagraphStyle("H", parent=styles["Heading2"], fontSize=12, spaceBefore=10, spaceAfter=4)
    b = ParagraphStyle("B", parent=styles["Normal"], fontSize=10, leading=14, alignment=TA_JUSTIFY, spaceAfter=3)
    story = [Paragraph("Cinema 21 - Panduan dan Kebijakan", styles["Title"]), Spacer(1,0.3*cm)]
    for line in KNOWLEDGE.strip().split("\\n"):
        line = line.strip()
        if not line: story.append(Spacer(1,0.1*cm))
        elif line.startswith("=== "): story.append(Paragraph(line.strip("= ").strip(), h)); story.append(HRFlowable(width="100%",thickness=0.5))
        else: story.append(Paragraph(line, b))
    doc.build(story)
    print(f"PDF generated: {output_path}")
'''
RAG_PY = "import os, pickle\nimport numpy as np\n\nFAISS_INDEX = \"cinema_agent/faiss_index.bin\"\nCHUNKS_PATH = \"cinema_agent/faiss_chunks.pkl\"\nPDF_PATH    = \"cinema_agent/cinema_knowledge.pdf\"\n_index = None; _chunks = None; _model = None\n\ndef _get_model():\n    global _model\n    if _model is None:\n        from sentence_transformers import SentenceTransformer\n        _model = SentenceTransformer(\"all-MiniLM-L6-v2\")\n    return _model\n\ndef _chunk(text, size=250, overlap=40):\n    words = text.split(); chunks = []; i = 0\n    while i < len(words):\n        c = \" \".join(words[i:i+size])\n        if c.strip(): chunks.append(c.strip())\n        i += size - overlap\n    return chunks\n\ndef build_index(pdf_path=PDF_PATH):\n    import faiss\n    from pdfminer.high_level import extract_text\n    text   = extract_text(pdf_path)\n    chunks = _chunk(text)\n    model  = _get_model()\n    emb = model.encode(chunks, show_progress_bar=True, convert_to_numpy=True).astype(np.float32)\n    emb /= (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-9)\n    idx = faiss.IndexFlatIP(emb.shape[1]); idx.add(emb)\n    faiss.write_index(idx, FAISS_INDEX)\n    with open(CHUNKS_PATH,\"wb\") as f: pickle.dump(chunks, f)\n    print(f\"FAISS: {idx.ntotal} vectors\"); return idx, chunks\n\ndef load_index():\n    global _index, _chunks\n    if _index is None:\n        import faiss\n        _index = faiss.read_index(FAISS_INDEX)\n        with open(CHUNKS_PATH,\"rb\") as f: _chunks = pickle.load(f)\n    return _index, _chunks\n\ndef search(query, top_k=4):\n    idx, chunks = load_index(); model = _get_model()\n    q = model.encode([query], convert_to_numpy=True).astype(np.float32)\n    q /= (np.linalg.norm(q, axis=1, keepdims=True) + 1e-9)\n    _, indices = idx.search(q, top_k)\n    return [chunks[i] for i in indices[0] if i < len(chunks)]\n\ndef init_rag():\n    if not os.path.exists(FAISS_INDEX): build_index(PDF_PATH)\n    load_index(); print(\"RAG ready.\")\n"
TOOLS_PY = '''from database import get_conn
import rag

def rag_search(query):
    try:
        chunks = rag.search(query, top_k=4)
        return "\\n\\n---\\n\\n".join(chunks) if chunks else "Tidak ditemukan informasi relevan."
    except Exception as e: return f"Error: {e}"

def get_movie_info(title):
    try:
        chunks = rag.search(f"sinopsis {title} cocok untuk", top_k=4)
        return "\\n\\n---\\n\\n".join(chunks) if chunks else "Informasi film tidak ditemukan."
    except Exception as e: return f"Error: {e}"

def search_movies(title="", genre=""):
    conn = get_conn()
    try:
        q = "SELECT id,title,genre,duration,age_rating,description FROM movies WHERE 1=1"
        params = []
        if title: q += " AND title LIKE ?"; params.append(f"%{title}%")
        if genre: q += " AND genre LIKE ?"; params.append(f"%{genre}%")
        rows = conn.execute(q, params).fetchall()
        if not rows: return "Tidak ada film yang cocok."
        return "\\n\\n".join(
            f"[{r['id']}] {r['title']}\\n   Genre: {r['genre']} | {r['duration']} menit | {r['age_rating']}\\n   {r['description']}"
            for r in rows)
    finally: conn.close()

def get_showtimes(movie_title="", show_date="", movie_id=0):
    from datetime import date
    from collections import defaultdict
    conn = get_conn()
    try:
        if not show_date: show_date = date.today().isoformat()
        q = "SELECT s.id,m.title,st.studio_name,st.type,s.show_date,s.show_time,s.price FROM showtimes s JOIN movies m ON s.movie_id=m.id JOIN studios st ON s.studio_id=st.id WHERE s.show_date=?"
        params = [show_date]
        if movie_id: q += " AND m.id=?"; params.append(movie_id)
        elif movie_title: q += " AND m.title LIKE ?"; params.append(f"%{movie_title}%")
        q += " ORDER BY m.title,s.show_time"
        rows = conn.execute(q, params).fetchall()
        if not rows: return f"Tidak ada jadwal untuk {show_date}."
        grouped = defaultdict(list)
        for r in rows: grouped[r['title']].append(r)
        out = [f"Jadwal - {show_date}"]
        for t, shows in grouped.items():
            out.append(f"\\n{t}")
            for s in shows: out.append(f"   [ID:{s['id']}] {s['show_time']} | {s['studio_name']} | Rp {s['price']:,}")
        return "\\n".join(out)
    finally: conn.close()

def check_seat_availability(showtime_id):
    conn = get_conn()
    try:
        show = conn.execute(
            "SELECT s.id,m.title,st.studio_name,s.show_date,s.show_time,s.price,st.id as studio_id FROM showtimes s JOIN movies m ON s.movie_id=m.id JOIN studios st ON s.studio_id=st.id WHERE s.id=?",
            (showtime_id,)).fetchone()
        if not show: return f"Showtime ID {showtime_id} tidak ditemukan."
        all_seats = [r['seat_number'] for r in conn.execute(
            "SELECT seat_number FROM seats WHERE studio_id=? ORDER BY seat_number",
            (show['studio_id'],)).fetchall()]
        booked = {r['seat_number'] for r in conn.execute(
            "SELECT seat_number FROM bookings WHERE showtime_id=? AND status='confirmed'",
            (showtime_id,)).fetchall()}
        available = [s for s in all_seats if s not in booked]
        if not available: return f"{show['title']} - KURSI PENUH."
        preview = ", ".join(available[:20])
        more = f" (+{len(available)-20} lainnya)" if len(available) > 20 else ""
        return (f"{show['title']}\\n"
                f"   {show['show_date']} {show['show_time']} | {show['studio_name']}\\n"
                f"   Harga: Rp {show['price']:,}\\n"
                f"   Tersedia: {len(available)}/{len(all_seats)} kursi\\n"
                f"   Kursi: {preview}{more}")
    finally: conn.close()

def create_booking(customer_name, showtime_id, seat_number):
    conn = get_conn()
    try:
        show = conn.execute(
            "SELECT s.id,m.title,st.studio_name,s.show_date,s.show_time,s.price FROM showtimes s JOIN movies m ON s.movie_id=m.id JOIN studios st ON s.studio_id=st.id WHERE s.id=?",
            (showtime_id,)).fetchone()
        if not show: return f"Showtime ID {showtime_id} tidak ditemukan."
        if conn.execute(
            "SELECT id FROM bookings WHERE showtime_id=? AND seat_number=? AND status='confirmed'",
            (showtime_id, seat_number.upper())).fetchone():
            return f"Kursi {seat_number.upper()} sudah dipesan."
        sid = conn.execute(
            "SELECT studio_id FROM showtimes WHERE id=?",
            (showtime_id,)).fetchone()['studio_id']
        if not conn.execute(
            "SELECT id FROM seats WHERE studio_id=? AND seat_number=?",
            (sid, seat_number.upper())).fetchone():
            return f"Kursi {seat_number.upper()} tidak valid."
        cur = conn.execute(
            "INSERT INTO bookings (customer_name,showtime_id,seat_number,status) VALUES (?,?,?,?)",
            (customer_name, showtime_id, seat_number.upper(), "confirmed"))
        conn.commit()
        return (f"BOOKING BERHASIL!\\n"
                f"   Booking ID: #{cur.lastrowid}\\n"
                f"   Nama: {customer_name}\\n"
                f"   Film: {show['title']}\\n"
                f"   {show['show_date']} {show['show_time']} | {show['studio_name']}\\n"
                f"   Kursi: {seat_number.upper()} | Rp {show['price']:,}\\n"
                f"   Harap tiba 15 menit sebelum film!")
    except Exception as e: return f"Gagal booking: {e}"
    finally: conn.close()

TOOL_FUNCTIONS = {"rag_search":rag_search,"get_movie_info":get_movie_info,"search_movies":search_movies,"get_showtimes":get_showtimes,"check_seat_availability":check_seat_availability,"create_booking":create_booking}
TOOL_DECLARATIONS = [
    {"name":"rag_search","description":"Cari info umum di PDF bioskop: kebijakan refund, aturan tiket, menu F&B, membership, promo, fasilitas, FAQ.","parameters":{"type":"object","properties":{"query":{"type":"string"}},"required":["query"]}},
    {"name":"get_movie_info","description":"Ambil sinopsis, ulasan, rekomendasi, dan info lengkap film. WAJIB dipanggil saat user tanya: film bagus tidak, cocok untuk siapa, rekomendasikan film, worth it tidak.","parameters":{"type":"object","properties":{"title":{"type":"string","description":"Judul film yang ingin dicari infonya"}},"required":["title"]}},
    {"name":"search_movies","description":"Cari film yang sedang tayang.","parameters":{"type":"object","properties":{"title":{"type":"string"},"genre":{"type":"string"}},"required":[]}},
    {"name":"get_showtimes","description":"Cek jadwal tayang. show_date format YYYY-MM-DD, kosong=hari ini.","parameters":{"type":"object","properties":{"movie_title":{"type":"string"},"show_date":{"type":"string"},"movie_id":{"type":"integer"}},"required":[]}},
    {"name":"check_seat_availability","description":"Cek kursi tersedia untuk showtime. Butuh showtime_id dari get_showtimes.","parameters":{"type":"object","properties":{"showtime_id":{"type":"integer"}},"required":["showtime_id"]}},
    {"name":"create_booking","description":"Buat pemesanan tiket. HANYA panggil setelah pengguna konfirmasi eksplisit.","parameters":{"type":"object","properties":{"customer_name":{"type":"string"},"showtime_id":{"type":"integer"},"seat_number":{"type":"string"}},"required":["customer_name","showtime_id","seat_number"]}},
]
'''
AGENT_PY = '''from datetime import date
import time, json
from google import genai
from google.genai import types
from tools import TOOL_DECLARATIONS, TOOL_FUNCTIONS

today = date.today()
day_name = today.strftime("%A")

SYSTEM_PROMPT = f"""Kamu adalah Asisten Bioskop Cinema 21 yang ramah dan profesional.
Hari ini: {today.isoformat()} ({day_name})

ATURAN:
1. JANGAN mengarang jadwal atau informasi yang tidak ada di database.
2. JANGAN mengarang promo atau menu — selalu ambil dari rag_search.
3. SELALU gunakan tools untuk data terkini.
4. Pertanyaan kebijakan/aturan/menu/promo -> rag_search.
5. Alur booking: cari film -> cek jadwal -> cek kursi -> inform promo -> konfirmasi -> booking.
6. SELALU minta konfirmasi eksplisit sebelum create_booking.
7. Jawab dalam Bahasa Indonesia yang ramah.
8. Pertanyaan "film bagus tidak", "worth it", "cocok untuk siapa", "rekomendasikan film" -> WAJIB panggil get_movie_info("<judul film>") SEBELUM menjawab. DILARANG menjawab tanpa memanggil get_movie_info terlebih dahulu.
9. Setelah booking berhasil -> rag_search("menu makanan minuman") lalu rekomendasikan snack yang cocok:
   * Film aksi/horor (Avatar, Venom, Dune) -> Combo 2 atau Nachos.
   * Film animasi/keluarga (Moana 2) -> Combo Pasangan atau Popcorn Caramel.
   * Film thriller/mind-bending (Inception) -> Kopi agar tetap fokus.
10. Pertanyaan "kursi mana yang bagus", "saran kursi", "rekomendasi tempat duduk" -> berikan saran langsung:
    * Regular/IMAX/4DX: Baris D-E kolom 3-6 = posisi terbaik, sejajar layar dan suara optimal.
    * Hindari baris A-B (terlalu dekat) dan baris H (terlalu jauh).
    * 4DX: posisi tengah lebih stabil saat kursi bergerak.
    * Premiere: semua kursi sofa recliner, bebas pilih.
    * JANGAN bilang tidak bisa memberikan saran — langsung rekomendasikan.

PROMO GUARDRAILS:
- Saat user mulai proses booking, SELALU panggil rag_search("promo diskon") terlebih dahulu.
- Informasikan HANYA promo yang relevan dengan kondisi berikut:
  * Hari ini {day_name} -> jika Selasa: sebut diskon member, jika Senin: sebut Senin Gembira
  * Tanyakan: apakah user pelajar/mahasiswa?
  * Tanyakan: apakah user punya kartu Mandiri/BCA?
  * Tanyakan: apakah user punya membership card Cinema 21?
- JANGAN sebut promo yang tidak relevan dengan hari ini.
- JANGAN sebut promo yang tidak ada di hasil rag_search.
- Jika tidak ada promo relevan, jangan sebut promo apapun.
"""

class CinemaAgent:
    def __init__(self, api_key):
        self.client   = genai.Client(api_key=api_key)
        self._api_key = api_key
        self.model    = "gemini-2.5-flash"
        self.history  = []
        self.tools    = [types.Tool(function_declarations=[types.FunctionDeclaration(**d) for d in TOOL_DECLARATIONS])]
        self.config   = types.GenerateContentConfig(system_instruction=SYSTEM_PROMPT, tools=self.tools, temperature=0.3)

    def chat(self, user_message):
        self.history.append(types.Content(role="user", parts=[types.Part(text=user_message)]))
        print(f"\\n{'='*50}\\nUSER: {user_message}\\n{'='*50}")
        for iteration in range(8):
            t0         = time.time()
            resp       = self.client.models.generate_content(model=self.model, contents=self.history, config=self.config)
            content    = resp.candidates[0].content
            tool_calls = [p.function_call for p in content.parts if p.function_call]
            text_parts = [p.text for p in content.parts if p.text]
            self.history.append(content)
            if not tool_calls:
                reply = "\\n".join(text_parts) or "(Tidak ada respons)"
                print(f"REPLY ({time.time()-t0:.2f}s): {reply[:120]}...")
                return reply
            for fc in tool_calls:
                args = dict(fc.args) if fc.args else {}
                print(f"  [{iteration+1}] TOOL: {fc.name}({json.dumps(args, ensure_ascii=False)})")
            results = []
            for fc in tool_calls:
                fn  = TOOL_FUNCTIONS.get(fc.name)
                t1  = time.time()
                res = fn(**(dict(fc.args) if fc.args else {})) if fn else f"Tool {fc.name} tidak ada"
                print(f"       RESULT ({time.time()-t1:.2f}s): {str(res)[:120]}...")
                results.append(types.Part(function_response=types.FunctionResponse(name=fc.name, response={"result": res})))
            self.history.append(types.Content(role="user", parts=results))
        return "Maaf, terjadi kesalahan. Silakan coba lagi."

    def reset(self): self.history = []
'''
MAIN_PY = "from contextlib import asynccontextmanager\nfrom fastapi import FastAPI, HTTPException\nfrom fastapi.middleware.cors import CORSMiddleware\nfrom pydantic import BaseModel\nimport sys\nsys.path.insert(0, '/content/cinema_agent')\n\nfrom database import init_db\nimport rag\n\n@asynccontextmanager\nasync def lifespan(app):\n    init_db(); rag.init_rag(); yield\n\napp = FastAPI(title=\"Cinema 21 Agent\", lifespan=lifespan)\napp.add_middleware(CORSMiddleware, allow_origins=[\"*\"], allow_methods=[\"*\"], allow_headers=[\"*\"])\n_sessions = {}\n\nclass ChatRequest(BaseModel):\n    session_id: str; message: str; api_key: str\n\nclass ChatResponse(BaseModel):\n    session_id: str; response: str\n\n@app.get(\"/health\")\ndef health(): return {\"status\": \"ok\"}\n\n@app.post(\"/chat\", response_model=ChatResponse)\ndef chat(req: ChatRequest):\n    from agent import CinemaAgent\n    if req.session_id not in _sessions or _sessions[req.session_id]._api_key != req.api_key:\n        _sessions[req.session_id] = CinemaAgent(api_key=req.api_key)\n    try:\n        reply = _sessions[req.session_id].chat(req.message)\n        return ChatResponse(session_id=req.session_id, response=reply)\n    except Exception as e:\n        raise HTTPException(status_code=500, detail=str(e))\n\n@app.delete(\"/session/{session_id}\")\ndef reset(session_id): _sessions.pop(session_id, None); return {\"status\": \"reset\"}\n"

files = {
    'cinema_agent/database.py':     DATABASE_PY,
    'cinema_agent/generate_pdf.py': GENERATE_PDF_PY,
    'cinema_agent/rag.py':          RAG_PY,
    'cinema_agent/tools.py':        TOOLS_PY,
    'cinema_agent/agent.py':        AGENT_PY,
    'main.py':                      MAIN_PY,
}
for path, content in files.items():
    with open(path, 'w', encoding='utf-8') as f: f.write(content.strip())
    print(f'OK {path}')
print('All files written.')

OK cinema_agent/database.py
OK cinema_agent/generate_pdf.py
OK cinema_agent/rag.py
OK cinema_agent/tools.py
OK cinema_agent/agent.py
OK main.py
All files written.


## Cell 4 - Write app.py (Streamlit UI)
Ditulis terpisah dengan %%writefile untuk menghindari masalah encoding emoji.

In [ ]:
%%writefile /content/app.py
import streamlit as st
import httpx, uuid, os

API_URL = os.environ.get("CINEMA_API_URL", "http://localhost:8000")
st.set_page_config(page_title="Cinema 21 AI", page_icon="🎬", layout="centered")

if "session_id" not in st.session_state: st.session_state.session_id = str(uuid.uuid4())
if "messages"   not in st.session_state: st.session_state.messages   = []
if "api_key"    not in st.session_state: st.session_state.api_key    = ""

with st.sidebar:
    st.markdown("## 🎬 Cinema 21")
    st.markdown("**AI Customer Service**")
    st.divider()
    key = st.text_input("Google AI API Key", type="password", value=st.session_state.api_key)
    if key: st.session_state.api_key = key
    st.divider()
    if st.button("🔄 Reset Percakapan", use_container_width=True):
        try: httpx.delete(f"{API_URL}/session/{st.session_state.session_id}", timeout=5)
        except: pass
        st.session_state.session_id = str(uuid.uuid4())
        st.session_state.messages   = []
        st.rerun()
    st.divider()
    st.markdown("**Capabilities:**")
    st.markdown("- 🎥 Info & jadwal film\n- 🎟️ Pemesanan tiket\n- 💺 Cek kursi\n- 📜 Kebijakan\n- 🍿 Menu F&B\n- 🎁 Promo & membership")

st.title("🎬 Cinema 21 AI Assistant")
st.caption("Asisten virtual Cinema 21 — tanya apa saja!")

if not st.session_state.api_key:
    st.info("👈 Masukkan Google AI API Key di sidebar.", icon="🔑")
    st.stop()

def send_message(msg):
    try:
        r = httpx.post(f"{API_URL}/chat",
                       json={"session_id": st.session_state.session_id,
                             "message": msg, "api_key": st.session_state.api_key},
                       timeout=90)
        return r.json()["response"] if r.status_code == 200 else f"Error: {r.json().get('detail')}"
    except Exception as e: return f"Error: {e}"

if not st.session_state.messages:
    st.markdown("**Coba tanya:**")
    suggestions = [
        ("Film apa yang tayang?",  "Film apa yang sedang tayang hari ini?"),
        ("Pesan tiket",            "Saya ingin memesan tiket film"),
        ("Kebijakan refund",       "Bagaimana kebijakan refund tiket?"),
        ("Menu makanan",           "Apa saja menu makanan yang tersedia?"),
        ("Ada promo?",             "Promo apa yang sedang berlaku?"),
        ("Info studio IMAX",       "Apa keuntungan nonton di studio IMAX?"),
    ]
    cols = st.columns(2)
    for i, (label, msg) in enumerate(suggestions):
        with cols[i % 2]:
            if st.button(label, key=f"s{i}", use_container_width=True):
                st.session_state.messages.append({"role":"user","content":msg})
                with st.spinner("Memproses..."): reply = send_message(msg)
                st.session_state.messages.append({"role":"assistant","content":reply})
                st.rerun()

for m in st.session_state.messages:
    with st.chat_message(m["role"]): st.markdown(m["content"])

if prompt := st.chat_input("Tanya tentang film, jadwal, booking, promo..."):
    st.session_state.messages.append({"role":"user","content":prompt})
    with st.chat_message("user"): st.markdown(prompt)
    with st.chat_message("assistant"):
        with st.spinner("Asisten sedang memproses..."): reply = send_message(prompt)
        st.markdown(reply)
    st.session_state.messages.append({"role":"assistant","content":reply})


Writing /content/app.py


## Cell 5 - Initialize Database, PDF & FAISS Index

In [ ]:
import sys, os
sys.path.insert(0, '/content/cinema_agent')

from generate_pdf import generate_pdf
generate_pdf()

from database import init_db
init_db()

import rag
for f in ['cinema_agent/faiss_index.bin','cinema_agent/faiss_chunks.pkl']:
    if os.path.exists(f): os.remove(f)
rag.build_index()
rag.load_index()
print('All systems ready!')

PDF generated: cinema_agent/cinema_knowledge.pdf
DB ready.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

FAISS: 4 vectors
All systems ready!


## Cell 6 - Start FastAPI (background thread)

In [ ]:
import threading, uvicorn, sys, os, time, importlib.util
sys.path.insert(0, '/content/cinema_agent')

def run_api():
    spec = importlib.util.spec_from_file_location('main', '/content/main.py')
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    uvicorn.run(mod.app, host='0.0.0.0', port=8000, log_level='warning')

t = threading.Thread(target=run_api, daemon=True)
t.start()
time.sleep(4)

import httpx
r = httpx.get('http://localhost:8000/health', timeout=10)
print('FastAPI status:', r.json())

DB ready.
RAG ready.
FastAPI status: {'status': 'ok'}


## Cell 7 - Start Streamlit + ngrok
Klik URL yang muncul, masukkan Gemini API key di sidebar Streamlit.

In [ ]:
import subprocess, time, os
from pyngrok import ngrok

os.environ['CINEMA_API_URL'] = 'http://localhost:8000'

subprocess.run(['pkill','-f','streamlit'], capture_output=True)
ngrok.kill()
time.sleep(2)

proc = subprocess.Popen(
    ['streamlit','run','/content/app.py',
     '--server.headless=true','--server.port','8501','--server.enableCORS=false'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

time.sleep(4)
public_url = ngrok.connect(8501)
print(f'Cinema 21 Agent: {public_url}')
print('Masukkan Google AI API key di sidebar Streamlit.')

Cinema 21 Agent: NgrokTunnel: "https://be28-34-87-150-9.ngrok-free.app" -> "http://localhost:8501"
Masukkan Google AI API key di sidebar Streamlit.


## Cell 8 - Stop

In [ ]:
import subprocess
from pyngrok import ngrok

try: proc.terminate(); print('Streamlit stopped.')
except: pass
subprocess.run(['pkill','-f','streamlit'], capture_output=True)
ngrok.kill()
print('All stopped.')

Streamlit stopped.
All stopped.


In [ ]:
import sqlite3
conn = sqlite3.connect('cinema_agent/cinema.db')
conn.row_factory = sqlite3.Row

# cek tables
tables = conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
print("Tables:", [t['name'] for t in tables])

# cek movies
print("\n--- MOVIES ---")
for r in conn.execute("SELECT * FROM movies").fetchall():
    print(dict(r))

# cek showtimes
print("\n--- SHOWTIMES (5 pertama) ---")
for r in conn.execute("""
    SELECT s.id, m.title, st.studio_name, s.show_date, s.show_time, s.price
    FROM showtimes s
    JOIN movies m ON s.movie_id = m.id
    JOIN studios st ON s.studio_id = st.id
    LIMIT 5
""").fetchall():
    print(dict(r))

# cek bookings
print("\n--- BOOKINGS ---")
rows = conn.execute("SELECT * FROM bookings").fetchall()
print(f"Total bookings: {len(rows)}")
for r in rows:
    print(dict(r))

conn.close()

Tables: ['movies', 'sqlite_sequence', 'cinemas', 'studios', 'showtimes', 'seats', 'bookings']

--- MOVIES ---
{'id': 1, 'title': 'Avatar: Fire and Ash', 'genre': 'Sci-Fi/Action', 'duration': 162, 'age_rating': '13+', 'description': 'Jake Sully dan Neytiri kembali.'}
{'id': 2, 'title': 'Inception 2: Dreamfall', 'genre': 'Thriller/Sci-Fi', 'duration': 148, 'age_rating': '17+', 'description': 'Dom Cobb kembali ke dunia mimpi.'}
{'id': 3, 'title': 'Moana 2', 'genre': 'Animation/Adventure', 'duration': 100, 'age_rating': 'SU', 'description': 'Moana memulai perjalanan baru.'}
{'id': 4, 'title': 'Venom: The Last Dance', 'genre': 'Action/Superhero', 'duration': 139, 'age_rating': '13+', 'description': 'Eddie Brock pertempuran terakhir.'}
{'id': 5, 'title': 'Dune: Messiah', 'genre': 'Sci-Fi/Epic', 'duration': 155, 'age_rating': '13+', 'description': 'Paul Atreides menjadi Kaisar.'}

--- SHOWTIMES (5 pertama) ---
{'id': 1, 'title': 'Avatar: Fire and Ash', 'studio_name': 'Studio 1', 'show_date': 

In [ ]:
import faiss, pickle
import numpy as np

# load index
index = faiss.read_index('cinema_agent/faiss_index.bin')
with open('cinema_agent/faiss_chunks.pkl', 'rb') as f:
    chunks = pickle.load(f)

# basic info
print(f"Total vectors : {index.ntotal}")
print(f"Dimension     : {index.d}")
print(f"Total chunks  : {len(chunks)}")

# lihat isi chunks (teks aslinya)
print("\n--- CHUNKS ---")
for i, chunk in enumerate(chunks):
    print(f"\n[{i}] {chunk[:200]}")

Total vectors : 4
Dimension     : 384
Total chunks  : 4

--- CHUNKS ---

[0] Cinema 21 - Panduan dan Kebijakan INFORMASI UMUM BIOSKOP Cinema 21 Grand Indonesia berlokasi di Grand Indonesia Mall Lantai 3A, Jakarta Pusat. Bioskop beroperasi setiap hari mulai pukul 10.00 WIB hing

[1] Rp 30.000 / L Rp 38.000. Air Mineral Rp 15.000. Nachos Rp 40.000. Hot Dog Rp 45.000. French Fries Rp 35.000. Combo 1 (Popcorn M + Soft Drink M) Rp 55.000. Combo 2 (L+L) Rp 70.000. Makanan dari luar TI

[2] antara manusia dan Na'vi memasuki babak baru yang lebih intens dan emosional. Cocok untuk: Pecinta aksi, visual spektakuler, science fiction. Sangat direkomendasikan di studio IMAX atau 4DX. Tidak coc

[3] Atreides kini menjadi Kaisar Arrakis namun kekuasaan membawa harga yang tidak terduga. Film epik dengan visual dan cerita yang sangat mendalam. Cocok untuk: Penggemar Dune Part 1 dan 2, sci-fi serius,


In [ ]:
import sqlite3
conn = sqlite3.connect('cinema_agent/cinema.db')
conn.row_factory = sqlite3.Row

print("--- STUDIOS ---")
for r in conn.execute("SELECT * FROM studios").fetchall():
    print(dict(r))

print("\n--- SHOWTIMES sample ---")
for r in conn.execute("""
    SELECT s.id, m.title, st.studio_name, st.type, s.show_time, s.price
    FROM showtimes s
    JOIN movies m ON s.movie_id=m.id
    JOIN studios st ON s.studio_id=st.id
    LIMIT 10
""").fetchall():
    print(dict(r))

--- STUDIOS ---
{'id': 1, 'cinema_id': 1, 'studio_name': 'Studio 1', 'type': 'Regular'}
{'id': 2, 'cinema_id': 1, 'studio_name': 'Studio 2', 'type': 'Regular'}
{'id': 3, 'cinema_id': 1, 'studio_name': 'Studio 3 - IMAX', 'type': 'IMAX'}
{'id': 4, 'cinema_id': 1, 'studio_name': 'Studio 4 - 4DX', 'type': '4DX'}
{'id': 5, 'cinema_id': 1, 'studio_name': 'Studio 5 - Premiere', 'type': 'Premiere'}

--- SHOWTIMES sample ---
{'id': 1, 'title': 'Avatar: Fire and Ash', 'studio_name': 'Studio 1', 'type': 'Regular', 'show_time': '11:00', 'price': 50000}
{'id': 2, 'title': 'Avatar: Fire and Ash', 'studio_name': 'Studio 2', 'type': 'Regular', 'show_time': '14:00', 'price': 50000}
{'id': 3, 'title': 'Avatar: Fire and Ash', 'studio_name': 'Studio 3 - IMAX', 'type': 'IMAX', 'show_time': '17:00', 'price': 100000}
{'id': 4, 'title': 'Inception 2: Dreamfall', 'studio_name': 'Studio 1', 'type': 'Regular', 'show_time': '11:00', 'price': 50000}
{'id': 5, 'title': 'Inception 2: Dreamfall', 'studio_name': 'Stud

In [ ]:
import sqlite3
conn = sqlite3.connect('cinema_agent/cinema.db')

for r in conn.execute("SELECT id, movie_id, studio_id, show_time, price FROM showtimes LIMIT 6").fetchall():
    print(tuple(r))

(1, 1, 1, '11:00', 50000)
(2, 1, 2, '14:00', 50000)
(3, 1, 3, '17:00', 100000)
(4, 2, 1, '11:00', 50000)
(5, 2, 2, '14:00', 50000)
(6, 2, 3, '17:00', 100000)


In [ ]:
import os, sys, sqlite3

# 1. hapus DB
os.remove('cinema_agent/cinema.db')
print("DB deleted")

# 2. clear module cache
for mod in list(sys.modules.keys()):
    if 'database' in mod:
        del sys.modules[mod]

# 3. reimport dan init
sys.path.insert(0, '/content/cinema_agent')
import database
database.init_db()

# 4. verify
conn = sqlite3.connect('cinema_agent/cinema.db')
for r in conn.execute("SELECT id, movie_id, studio_id, show_time, price FROM showtimes LIMIT 6").fetchall():
    print(tuple(r))
conn.close()

DB deleted
Seed done.
DB ready.
(1, 1, 1, '11:00', 50000)
(2, 1, 2, '14:00', 50000)
(3, 1, 3, '17:00', 100000)
(4, 2, 1, '11:00', 50000)
(5, 2, 2, '14:00', 50000)
(6, 2, 3, '17:00', 100000)


In [ ]:
import os, sys, importlib
sys.path.insert(0, '/content/cinema_agent')

import generate_pdf; importlib.reload(generate_pdf)
generate_pdf.generate_pdf()

import rag; importlib.reload(rag)
for f in ['cinema_agent/faiss_index.bin','cinema_agent/faiss_chunks.pkl']:
    if os.path.exists(f): os.remove(f)
rag.build_index()
rag.load_index()
print("Done")

PDF generated: cinema_agent/cinema_knowledge.pdf


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

FAISS: 4 vectors
Done


In [ ]:
import pickle
with open('cinema_agent/faiss_chunks.pkl', 'rb') as f:
    chunks = pickle.load(f)

# cari chunk yang ada kata sinopsis
for i, c in enumerate(chunks):
    if 'sinopsis' in c.lower() or 'avatar' in c.lower():
        print(f"[{i}] {c[:200]}")
        print("---")


[1] Rp 30.000 / L Rp 38.000. Air Mineral Rp 15.000. Nachos Rp 40.000. Hot Dog Rp 45.000. French Fries Rp 35.000. Combo 1 (Popcorn M + Soft Drink M) Rp 55.000. Combo 2 (L+L) Rp 70.000. Makanan dari luar TI
---
[2] antara manusia dan Na'vi memasuki babak baru yang lebih intens dan emosional. Cocok untuk: Pecinta aksi, visual spektakuler, science fiction. Sangat direkomendasikan di studio IMAX atau 4DX. Tidak coc
---


In [ ]:
import httpx, uuid

r = httpx.post(
    "http://localhost:8000/chat",
    json={
        "session_id": str(uuid.uuid4()),
        "message": "Film apa yang sedang tayang hari ini?",
        "api_key": ''
    },
    timeout=60
)
print("Status:", r.status_code)
print("Raw:", r.text[:500])

ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 421, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 62, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1159, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

Status: 500
Raw: Internal Server Error


In [ ]:
import subprocess, time, sys, threading, uvicorn, importlib

# 1. kill semua
subprocess.run(['pkill', '-f', 'uvicorn'], capture_output=True)
time.sleep(3)

# 2. clear cache
for mod in list(sys.modules.keys()):
    if any(x in mod for x in ['agent','tools','rag','database','main']):
        del sys.modules[mod]

# 3. verify model
with open('/content/cinema_agent/agent.py') as f:
    content = f.read()
print("Model:", "gemini-2.0-flash" if "gemini-2.0-flash" in content else "SALAH - " + [l for l in content.split('\n') if 'self.model' in l][0])

# 4. restart
def run_api():
    sys.path.insert(0, '/content/cinema_agent')
    spec = importlib.util.spec_from_file_location('main', '/content/main.py')
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    uvicorn.run(mod.app, host='0.0.0.0', port=8000, log_level='warning')

t = threading.Thread(target=run_api, daemon=True)
t.start()
time.sleep(5)

import httpx
r = httpx.get('http://localhost:8000/health', timeout=10)
print('FastAPI:', r.json())

Model: gemini-2.0-flash


ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use


DB ready.
RAG ready.
FastAPI: {'status': 'ok'}


In [ ]:
import httpx, uuid

# reset session lama
httpx.delete('http://localhost:8000/session/test', timeout=5)

# test dengan session baru
r = httpx.post(
    'http://localhost:8000/chat',
    json={
        'session_id': str(uuid.uuid4()),  # session ID baru
        'message': 'Film apa yang tayang hari ini?',
        'api_key': ''
    },
    timeout=60
)
print('Status:', r.status_code)
print('Response:', r.text[:300])


USER: Film apa yang tayang hari ini?
Status: 500
Response: {"detail":"429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/r


In [ ]:
from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))
for m in client.models.list():
    if 'flash' in m.name.lower():
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.5-flash
models/gemini-3.1-flash-tts-preview
models/gemini-2.5-flash-native-audio-latest
models/gemini-2.5-flash-native-audio-preview-09-2025
models/gemini-2.5-flash-native-audio-preview-12-2025
models/gemini-3.1-flash-live-preview
